In [2]:
import pandas as pd
import glob
import os
import requests
import rasterio
import numpy as np
import json
import time

In [3]:

judete = {
    "Alba":             (46.07, 23.57),
    "Arad":             (46.17, 21.32),
    "Arges":            (45.08, 24.87),
    "Bacau":            (46.57, 26.91),
    "Bihor":            (47.05, 21.95),
    "Bistrita-Nasaud":  (47.13, 24.50),
    "Botosani":         (47.75, 26.67),
    "Brasov":           (45.65, 25.61),
    "Braila":           (45.27, 27.96),
    "Buzau":            (45.15, 26.82),
    "Caras-Severin":    (45.30, 22.07),
    "Calarasi":         (44.20, 26.33),
    "Cluj":             (46.77, 23.59),
    "Constanta":        (44.18, 28.65),
    "Covasna":          (45.87, 26.18),
    "Dambovita":        (44.93, 25.45),
    "Dolj":             (44.31, 23.80),
    "Galati":           (45.43, 28.05),
    "Giurgiu":          (43.90, 25.97),
    "Gorj":             (44.95, 23.27),
    "Harghita":         (46.43, 25.57),
    "Hunedoara":        (45.75, 22.90),
    "Ialomita":         (44.60, 27.37),
    "Iasi":             (47.16, 27.59),
    "Ilfov":            (44.53, 26.17),
    "Maramures":        (47.66, 23.57),
    "Mehedinti":        (44.63, 22.65),
    "Mures":            (46.55, 24.56),
    "Neamt":            (46.97, 26.38),
    "Olt":              (44.43, 24.37),
    "Prahova":          (45.07, 25.99),
    "Satu Mare":        (47.80, 22.87),
    "Salaj":            (47.19, 23.06),
    "Sibiu":            (45.80, 24.15),
    "Suceava":          (47.65, 26.25),
    "Teleorman":        (43.98, 25.00),
    "Timis":            (45.75, 21.22),
    "Tulcea":           (45.18, 29.02),
    "Vaslui":           (46.64, 27.73),
    "Valcea":           (45.10, 24.37),
    "Vrancea":          (45.70, 27.18),
    "Bucuresti":        (44.43, 26.10),
}

PARAMETERS = [
    "T2M",
    "T2M_MIN",
    "T2M_MAX",
    "T2MDEW",
    "PRECTOTCORR",
    "EVPTRNS",
    "ALLSKY_SFC_SW_DWN",
    "ALLSKY_SFC_PAR_TOT",
    "CDD0",
    "CDD10",
    "HDD0",
    "RH2M",
    "WS2M",
    "FROST_DAYS",
    "GWETTOP",
    "GWETROOT",
]

sezoane = {
    "iarna":     [1, 2],
    "primavara": [3, 4, 5],
    "vara":      [6, 7, 8],
    "toamna":    [9, 10, 11],
}

def fetch_nasa_judet(judet, lat, lon, an_start=2000, an_end=2024):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": ",".join(PARAMETERS),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": f"{an_start}0101",
        "end": f"{an_end}1231",
        "format": "JSON"
    }

    for attempt in range(5):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                print(f"  [{judet}] Rate limit, astept 65s (incercare {attempt+1}/5)...")
                time.sleep(65)
            else:
                print(f"  [{judet}] HTTP {r.status_code}: {r.text[:200]}")
                return None
        except Exception as e:
            print(f"  [{judet}] Eroare: {e}")
            return None
    else:
        print(f"  [{judet}] ESUAT dupa 5 incercari")
        return None

    data = r.json()
    daily = data["properties"]["parameter"]

    dates = list(daily["T2M"].keys())
    df = pd.DataFrame({"date": pd.to_datetime(dates, format="%Y%m%d")})
    df["An"] = df["date"].dt.year
    df["luna"] = df["date"].dt.month

    for param in PARAMETERS:
        if param in daily:
            df[param] = list(daily[param].values())
            df[param] = df[param].replace(-999, float("nan"))

    rows = []
    for an in range(an_start, an_end + 1):
        df_an = df[df["An"] == an].copy()
        row = {"Judet": judet, "An": an}

        # agregare anuala
        row["temp_medie_an"]      = round(df_an["T2M"].mean(), 2)
        row["temp_min_an"]        = round(df_an["T2M_MIN"].min(), 2)
        row["temp_max_an"]        = round(df_an["T2M_MAX"].max(), 2)
        row["precipitatii_an"]    = round(df_an["PRECTOTCORR"].sum(), 1)
        row["zile_inghet_an"]     = round(df_an["FROST_DAYS"].sum(), 0)
        row["radiatie_an"]        = round(df_an["ALLSKY_SFC_SW_DWN"].sum(), 1)
        row["par_an"]             = round(df_an["ALLSKY_SFC_PAR_TOT"].sum(), 1)
        row["umiditate_an"]       = round(df_an["RH2M"].mean(), 2)
        row["evapotrans_an"]      = round(df_an["EVPTRNS"].sum(), 1)
        row["cdd10_an"]           = round(df_an["CDD10"].sum(), 1)
        row["cdd0_an"]            = round(df_an["CDD0"].sum(), 1)
        row["hdd0_an"]            = round(df_an["HDD0"].sum(), 1)
        row["vant_an"]            = round(df_an["WS2M"].mean(), 2)
        row["umid_sol_sup_an"]    = round(df_an["GWETTOP"].mean(), 3)
        row["umid_sol_rad_an"]    = round(df_an["GWETROOT"].mean(), 3)

        # agregare pe sezoane
        for sezon, luni in sezoane.items():
            df_s = df_an[df_an["luna"].isin(luni)]
            row[f"temp_medie_{sezon}"]    = round(df_s["T2M"].mean(), 2)
            row[f"temp_min_{sezon}"]      = round(df_s["T2M_MIN"].min(), 2)
            row[f"temp_max_{sezon}"]      = round(df_s["T2M_MAX"].max(), 2)
            row[f"precipitatii_{sezon}"]  = round(df_s["PRECTOTCORR"].sum(), 1)
            row[f"zile_inghet_{sezon}"]   = round(df_s["FROST_DAYS"].sum(), 0)
            row[f"radiatie_{sezon}"]      = round(df_s["ALLSKY_SFC_SW_DWN"].sum(), 1)
            row[f"umiditate_{sezon}"]     = round(df_s["RH2M"].mean(), 2)
            row[f"evapotrans_{sezon}"]    = round(df_s["EVPTRNS"].sum(), 1)
            row[f"cdd10_{sezon}"]         = round(df_s["CDD10"].sum(), 1)
            row[f"umid_sol_sup_{sezon}"]  = round(df_s["GWETTOP"].mean(), 3)
            row[f"umid_sol_rad_{sezon}"]  = round(df_s["GWETROOT"].mean(), 3)

        rows.append(row)

    return pd.DataFrame(rows)


results = []
total = len(judete)

for i, (judet, (lat, lon)) in enumerate(judete.items(), 1):
    print(f"[{i}/{total}] {judet}...")
    df_judet = fetch_nasa_judet(judet, lat, lon)
    if df_judet is not None:
        results.append(df_judet)
        print(f"  OK — {len(df_judet)} ani")
    time.sleep(3)

df_clima = pd.concat(results, ignore_index=True)
df_clima.to_csv("clima_nasa_judete_romania.csv", index=False)
print(f"\nSalvat: {df_clima.shape[0]} randuri, {df_clima.shape[1]} coloane")
print(df_clima.head())

[1/42] Alba...
  OK — 25 ani
[2/42] Arad...
  OK — 25 ani
[3/42] Arges...
  OK — 25 ani
[4/42] Bacau...
  OK — 25 ani
[5/42] Bihor...
  OK — 25 ani
[6/42] Bistrita-Nasaud...
  OK — 25 ani
[7/42] Botosani...
  OK — 25 ani
[8/42] Brasov...
  OK — 25 ani
[9/42] Braila...
  OK — 25 ani
[10/42] Buzau...
  OK — 25 ani
[11/42] Caras-Severin...
  OK — 25 ani
[12/42] Calarasi...
  OK — 25 ani
[13/42] Cluj...
  OK — 25 ani
[14/42] Constanta...
  OK — 25 ani
[15/42] Covasna...
  OK — 25 ani
[16/42] Dambovita...
  OK — 25 ani
[17/42] Dolj...
  OK — 25 ani
[18/42] Galati...
  OK — 25 ani
[19/42] Giurgiu...
  OK — 25 ani
[20/42] Gorj...
  OK — 25 ani
[21/42] Harghita...
  OK — 25 ani
[22/42] Hunedoara...
  OK — 25 ani
[23/42] Ialomita...
  OK — 25 ani
[24/42] Iasi...
  OK — 25 ani
[25/42] Ilfov...
  OK — 25 ani
[26/42] Maramures...
  OK — 25 ani
[27/42] Mehedinti...
  OK — 25 ani
[28/42] Mures...
  OK — 25 ani
[29/42] Neamt...
  OK — 25 ani
[30/42] Olt...
  OK — 25 ani
[31/42] Prahova...
  OK — 25 a

In [ ]:
df_master = pd.read_csv('date_meri_final.csv')
df_clima = pd.read_csv('clima_nasa_judete_romania.csv')

df_clima.head()
df_master.head()

,Judet,An,temp_medie_an,temp_min_an,temp_max_an,precipitatii_an,zile_inghet_an,radiatie_an,par_an,umiditate_an,...,temp_min_toamna,temp_max_toamna,precipitatii_toamna,zile_inghet_toamna,radiatie_toamna,umiditate_toamna,evapotrans_toamna,cdd10_toamna,umid_sol_sup_toamna,umid_sol_rad_toamna
0,Alba,2000,10.66,-20.17,37.89,253.6,104.0,4795.2,2329.6,62.20,...,-2.07,31.25,38.0,5.0,857.8,59.71,0.9,316.0,0.331,0.415
1,Alba,2001,9.78,-18.48,36.33,519.0,115.0,4323.0,1909.2,68.04,...,-8.92,28.87,138.6,24.0,722.7,72.23,11.5,287.3,0.464,0.475
2,Alba,2002,10.47,-15.25,36.92,414.9,104.0,4387.7,1930.0,66.59,...,-5.61,27.88,131.0,13.0,727.7,70.79,6.8,242.9,0.456,0.477
3,Alba,2003,9.38,-17.15,34.00,508.1,134.0,4755.3,2083.6,70.30,...,-5.45,30.29,195.1,18.0,806.7,76.24,15.5,205.3,0.548,0.533
4,Alba,2004,9.26,-16.84,36.34,706.4,114.0,4486.5,1983.6,74.83,...,-5.65,28.78,180.0,14.0,842.1,74.07,33.4,281.0,0.537,0.532
